In [2]:
# ============================================================
# МОДУЛЬ 3: DECISION FUSION - ОБУЧЕНИЕ
# ============================================================
# 
# Входы:
#   - /kaggle/working/results/scores_module1.csv (от WavLM+AASIST)
#   - /kaggle/working/results/scores_module2.csv (от Whisper+Prosody)
#
# Выход:
#   - /kaggle/working/models/fusion_model.pkl (обученная LogisticRegression)
#
# ============================================================

import os
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, 
    confusion_matrix, precision_score, recall_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print("=" * 80)
print("🔄 МОДУЛЬ 3: DECISION FUSION - ОБУЧЕНИЕ ЛОГИСТИЧЕСКОЙ РЕГРЕССИИ")
print("=" * 80)

# ============================================================
# 1️⃣ ЗАГРУЗИТЬ CSV FILES ОТ MODULE1 И MODULE2
# ============================================================

print("\n📥 ЭТАП 1: Загрузка CSV files...")
print("-" * 80)

CSV_M1_PATH = "/kaggle/input/module1-2/scores_module1.csv"
CSV_M2_PATH = "/kaggle/input/module1-2/scores_module2.csv"

# Проверка существования файлов
if not os.path.exists(CSV_M1_PATH):
    raise FileNotFoundError(f"Файл не найден: {CSV_M1_PATH}")
if not os.path.exists(CSV_M2_PATH):
    raise FileNotFoundError(f"Файл не найден: {CSV_M2_PATH}")

# Загрузить CSV
df_m1 = pd.read_csv(CSV_M1_PATH)
df_m2 = pd.read_csv(CSV_M2_PATH)

print(f"✅ Module1 CSV загружена:")
print(f"   Размер: {len(df_m1)} файлов")
print(f"   Колонки: {list(df_m1.columns)}")
print(f"   \n{df_m1.head()}")

print(f"\n✅ Module2 CSV загружена:")
print(f"   Размер: {len(df_m2)} файлов")
print(f"   Колонки: {list(df_m2.columns)}")
print(f"   \n{df_m2.head()}")

# ============================================================
# 2️⃣ ОБЪЕДИНИТЬ ДАТАСЕТЫ
# ============================================================

print("\n" + "=" * 80)
print("🔗 ЭТАП 2: Объединение датасетов...")
print("-" * 80)

# Переименовать для ясности
df_m1 = df_m1.rename(columns={
    'probability': 'score_deepfake'
})

df_m2 = df_m2.rename(columns={
    'keyword_score': 'score_keyword',
    'prosody_score': 'score_prosody'
})

# Объединить по filename и label
df_fusion = pd.merge(
    df_m1[['filename', 'score_deepfake', 'label']],
    df_m2[['filename', 'score_keyword', 'score_prosody', 'label']],
    on=['filename', 'label'],
    how='inner'
)

print(f"\n✅ Датасеты объединены:")
print(f"   Размер: {len(df_fusion)} примеров")
print(f"   Колонки: {list(df_fusion.columns)}")
print(f"   \n{df_fusion.head()}")

# Проверка на NaN
print(f"\n🔍 Проверка NaN:")
print(df_fusion.isnull().sum())

if df_fusion.isnull().any().any():
    print("⚠️  Найдены NaN значения, удаляю...")
    df_fusion = df_fusion.dropna()
    print(f"   После удаления: {len(df_fusion)} примеров")

# Статистика по классам
print(f"\n📊 Распределение классов:")
print(f"   Класс 0 (Normal): {len(df_fusion[df_fusion['label'] == 0])} примеров")
print(f"   Класс 1 (Fraud):  {len(df_fusion[df_fusion['label'] == 1])} примеров")
print(f"   Баланс: {len(df_fusion[df_fusion['label'] == 0]) / len(df_fusion) * 100:.1f}% / {len(df_fusion[df_fusion['label'] == 1]) / len(df_fusion) * 100:.1f}%")

# ============================================================
# 3️⃣ ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ
# ============================================================

print("\n" + "=" * 80)
print("🔧 ЭТАП 3: Подготовка данных...")
print("-" * 80)

# Извлечь признаки и таргет
X = df_fusion[['score_deepfake', 'score_keyword', 'score_prosody']].values
y = df_fusion['label'].values

print(f"\n📐 Входные признаки (X):")
print(f"   Форма: {X.shape}")
print(f"   Признаки: ['score_deepfake', 'score_keyword', 'score_prosody']")
print(f"   Диапазон каждого: [0, 1]")

print(f"\n📊 Целевая переменная (y):")
print(f"   Форма: {y.shape}")
print(f"   Классы: {np.unique(y)}")

# Статистика по признакам
print(f"\n📈 Статистика признаков:")
print(f"   score_deepfake:  mean={X[:, 0].mean():.3f}, std={X[:, 0].std():.3f}, min={X[:, 0].min():.3f}, max={X[:, 0].max():.3f}")
print(f"   score_keyword:   mean={X[:, 1].mean():.3f}, std={X[:, 1].std():.3f}, min={X[:, 1].min():.3f}, max={X[:, 1].max():.3f}")
print(f"   score_prosody:   mean={X[:, 2].mean():.3f}, std={X[:, 2].std():.3f}, min={X[:, 2].min():.3f}, max={X[:, 2].max():.3f}")

# ============================================================
# 4️⃣ СТАНДАРТИЗАЦИЯ ПРИЗНАКОВ
# ============================================================

print("\n" + "=" * 80)
print("⚖️  ЭТАП 4: Стандартизация признаков...")
print("-" * 80)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\n✅ Признаки стандартизированы:")
print(f"   mean={X_scaled.mean():.6f}, std={X_scaled.std():.6f}")
print(f"\n   После стандартизации:")
print(f"   score_deepfake:  mean={X_scaled[:, 0].mean():.3f}, std={X_scaled[:, 0].std():.3f}")
print(f"   score_keyword:   mean={X_scaled[:, 1].mean():.3f}, std={X_scaled[:, 1].std():.3f}")
print(f"   score_prosody:   mean={X_scaled[:, 2].mean():.3f}, std={X_scaled[:, 2].std():.3f}")

# ============================================================
# 5️⃣ ОБУЧЕНИЕ ЛОГИСТИЧЕСКОЙ РЕГРЕССИИ
# ============================================================

print("\n" + "=" * 80)
print("🚀 ЭТАП 5: Обучение Логистической Регрессии...")
print("-" * 80)

# Параметры
class_weights = {0: 1.0, 1: 1.0}  # Можно менять при дисбалансе
C_param = 1.0
max_iter = 10000
random_state = 42

print(f"\n⚙️  Гиперпараметры:")
print(f"   C (регуляризация): {C_param}")
print(f"   max_iter: {max_iter}")
print(f"   class_weights: {class_weights}")
print(f"   random_state: {random_state}")

# Создать и обучить модель
fusion_model = LogisticRegression(
    C=C_param,
    max_iter=max_iter,
    class_weight=class_weights,
    random_state=random_state,
    verbose=0
)

print(f"\n🔄 Обучение модели...")
fusion_model.fit(X_scaled, y)
print(f"✅ Модель обучена!")

# Веса модели (коэффициенты)
coefficients = fusion_model.coef_[0]
intercept = fusion_model.intercept_[0]

print(f"\n📊 Параметры обученной модели:")
print(f"   Коэффициенты (веса):")
print(f"      - score_deepfake:  {coefficients[0]:.4f}")
print(f"      - score_keyword:   {coefficients[1]:.4f}")
print(f"      - score_prosody:   {coefficients[2]:.4f}")
print(f"   Intercept (смещение): {intercept:.4f}")

# ============================================================
# 6️⃣ НОРМАЛИЗОВАННЫЕ ВЕСА (ДЛЯ ИНТЕРПРЕТАЦИИ)
# ============================================================

print("\n" + "=" * 80)
print("⚖️  ЭТАП 6: Нормализованные веса...")
print("-" * 80)

# Нормализовать веса (сумма = 1)
abs_coefficients = np.abs(coefficients)
normalized_weights = abs_coefficients / abs_coefficients.sum()

print(f"\n📊 Нормализованные веса (для интерпретации):")
print(f"   score_deepfake:  {normalized_weights[0]:.4f} ({normalized_weights[0]*100:.1f}%)")
print(f"   score_keyword:   {normalized_weights[1]:.4f} ({normalized_weights[1]*100:.1f}%)")
print(f"   score_prosody:   {normalized_weights[2]:.4f} ({normalized_weights[2]*100:.1f}%)")

print(f"\n💡 Интерпретация:")
print(f"   Deepfake детектор (WavLM) имеет вес {normalized_weights[0]*100:.1f}%")
print(f"   Ключевые слова (Whisper) имеют вес {normalized_weights[1]*100:.1f}%")
print(f"   Просодия имеет вес {normalized_weights[2]*100:.1f}%")

# ============================================================
# 7️⃣ ОЦЕНКА МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ
# ============================================================

print("\n" + "=" * 80)
print("📊 ЭТАП 7: Оценка модели на тренировочных данных...")
print("-" * 80)

# Предсказания
y_pred = fusion_model.predict(X_scaled)
y_pred_proba = fusion_model.predict_proba(X_scaled)[:, 1]  # Вероятность класса 1

# Метрики
accuracy = accuracy_score(y, y_pred)
precision = precision_score(y, y_pred, zero_division=0)
recall = recall_score(y, y_pred, zero_division=0)
f1 = f1_score(y, y_pred, zero_division=0)
auc = roc_auc_score(y, y_pred_proba)

cm = confusion_matrix(y, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"\n✅ МЕТРИКИ МОДЕЛИ:")
print(f"   Accuracy:   {accuracy:.4f}")
print(f"   Precision:  {precision:.4f}")
print(f"   Recall:     {recall:.4f}")
print(f"   F1-Score:   {f1:.4f}")
print(f"   ROC-AUC:    {auc:.4f}")
print(f"   Sensitivity: {sensitivity:.4f}")
print(f"   Specificity: {specificity:.4f}")

print(f"\n📋 Confusion Matrix:")
print(f"   TN={tn}, FP={fp}")
print(f"   FN={fn}, TP={tp}")

# ============================================================
# 8️⃣ СТАТИСТИЧЕСКИЕ ТЕСТЫ ДЛЯ ОПРЕДЕЛЕНИЯ THRESHOLD
# ============================================================

print("\n" + "=" * 80)
print("📈 ЭТАП 8: Статистические тесты для threshold...")
print("-" * 80)

# Разделить скоры по классам
fraud_probs = y_pred_proba[y == 1]
normal_probs = y_pred_proba[y == 0]

print(f"\n📊 Распределение вероятностей:")
print(f"   Normal (класс 0):")
print(f"      mean={normal_probs.mean():.3f}, std={normal_probs.std():.3f}")
print(f"      min={normal_probs.min():.3f}, max={normal_probs.max():.3f}")
print(f"      [Q1={np.percentile(normal_probs, 25):.3f}, Median={np.median(normal_probs):.3f}, Q3={np.percentile(normal_probs, 75):.3f}]")

print(f"\n   Fraud (класс 1):")
print(f"      mean={fraud_probs.mean():.3f}, std={fraud_probs.std():.3f}")
print(f"      min={fraud_probs.min():.3f}, max={fraud_probs.max():.3f}")
print(f"      [Q1={np.percentile(fraud_probs, 25):.3f}, Median={np.median(fraud_probs):.3f}, Q3={np.percentile(fraud_probs, 75):.3f}]")

# T-тест
t_stat, t_pvalue = stats.ttest_ind(fraud_probs, normal_probs)
print(f"\n🔬 T-тест (различие между классами):")
print(f"   t-statistic: {t_stat:.4f}")
print(f"   p-value: {t_pvalue:.6f}")
if t_pvalue < 0.05:
    print(f"   ✅ Классы статистически РАЗЛИЧАЮТСЯ (p < 0.05)")
else:
    print(f"   ❌ Классы НЕ различаются (p >= 0.05)")

# Оптимальный threshold (Youden's index)
thresholds_test = np.linspace(0, 1, 100)
youden_scores = []

for thresh in thresholds_test:
    y_pred_thresh = (y_pred_proba >= thresh).astype(int)
    cm_thresh = confusion_matrix(y, y_pred_thresh)
    if cm_thresh.shape == (2, 2):
        tn, fp, fn, tp = cm_thresh.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        youden = sens + spec - 1
        youden_scores.append(youden)
    else:
        youden_scores.append(0)

optimal_idx = np.argmax(youden_scores)
optimal_threshold = thresholds_test[optimal_idx]

print(f"\n🎯 Оптимальный threshold (Youden's Index):")
print(f"   Threshold: {optimal_threshold:.3f}")
print(f"   Youden Score: {youden_scores[optimal_idx]:.3f}")

# ============================================================
# 9️⃣ СОХРАНЕНИЕ МОДЕЛИ
# ============================================================

print("\n" + "=" * 80)
print("💾 ЭТАП 9: Сохранение модели...")
print("-" * 80)

MODEL_DIR = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "fusion_model.pkl")

# Создать dictionary с моделью и всеми параметрами
fusion_package = {
    'model': fusion_model,
    'scaler': scaler,
    'coefficients': coefficients,
    'intercept': intercept,
    'normalized_weights': normalized_weights,
    'optimal_threshold': optimal_threshold,
    'feature_names': ['score_deepfake', 'score_keyword', 'score_prosody'],
    'metrics': {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'sensitivity': sensitivity,
        'specificity': specificity
    },
    'statistics': {
        'fraud_mean': fraud_probs.mean(),
        'fraud_std': fraud_probs.std(),
        'normal_mean': normal_probs.mean(),
        'normal_std': normal_probs.std(),
        't_statistic': t_stat,
        't_pvalue': t_pvalue
    }
}

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(fusion_package, f)

print(f"\n✅ Модель сохранена!")
print(f"   Путь: {MODEL_PATH}")
print(f"   Размер: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB")

# ============================================================
# 🔟 ФИНАЛЬНАЯ СТАТИСТИКА
# ============================================================

print("\n" + "=" * 80)
print("✅ МОДУЛЬ 3 ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("=" * 80)

print(f"""
📦 СОХРАНЕНО В: {MODEL_PATH}

🔍 КРАТКИЕ ИТОГИ:
   ✅ Обучающий датасет: {len(df_fusion)} примеров
   ✅ Классы: Normal={len(df_fusion[df_fusion['label']==0])}, Fraud={len(df_fusion[df_fusion['label']==1])}
   
🎯 ПРОИЗВОДИТЕЛЬНОСТЬ МОДЕЛИ:
   ✅ Accuracy:   {accuracy:.4f}
   ✅ F1-Score:   {f1:.4f}
   ✅ ROC-AUC:    {auc:.4f}
   
⚖️  ВЕСА РЕШЕНИЙ:
   ✅ Deepfake:   {normalized_weights[0]*100:.1f}%
   ✅ Keywords:   {normalized_weights[1]*100:.1f}%
   ✅ Prosody:    {normalized_weights[2]*100:.1f}%
   
🎚️  ПОРОГИ:
   ✅ Optimal threshold: {optimal_threshold:.3f}
   ✅ (Можно менять вручную в Module 4)

📊 СТАТИСТИЧЕСКАЯ ЗНАЧИМОСТЬ:
   ✅ T-тест p-value: {t_pvalue:.6f}
   ✅ Классы {'РАЗЛИЧАЮТСЯ' if t_pvalue < 0.05 else 'НЕ РАЗЛИЧАЮТСЯ'} (p {'<' if t_pvalue < 0.05 else '>='} 0.05)

🚀 СЛЕДУЮЩИЙ ШАГ:
   → Создать Module4_INFERENCE.ipynb
   → Загрузить эту модель
   → Применить к новым аудио через DeepfakeInferenceModule + FraudPatternInferenceModule
   → Вывести красивый результат
""")

print("=" * 80)


🔄 МОДУЛЬ 3: DECISION FUSION - ОБУЧЕНИЕ ЛОГИСТИЧЕСКОЙ РЕГРЕССИИ

📥 ЭТАП 1: Загрузка CSV files...
--------------------------------------------------------------------------------
✅ Module1 CSV загружена:
   Размер: 200 файлов
   Колонки: ['filename', 'probability', 'label']
   
         filename   probability  label
0  real_54312.wav  2.708828e-06      0
1  real_42021.wav  4.757429e-07      0
2  real_59404.wav  6.318032e-06      0
3  real_11516.wav  5.583777e-08      0
4  real_34412.wav  2.699974e-06      0

✅ Module2 CSV загружена:
   Размер: 200 файлов
   Колонки: ['filename', 'keyword_score', 'prosody_score', 'label']
   
         filename  keyword_score  prosody_score  label
0  real_54312.wav            0.0       0.077825      0
1  real_42021.wav            0.0       0.695907      0
2  real_59404.wav            0.0       0.072432      0
3  real_11516.wav            0.0       0.722049      0
4  real_34412.wav            0.0       0.433602      0

🔗 ЭТАП 2: Объединение датасетов...
---